# 🤖 TinyLLM-nano Kaggle 継続学習ノートブック

**Kaggle GPU (T4 x2 / P100 16GB) で TinyLLM-nano (318M params) を継続事前学習します。**

## ✅ 全OOM/ディスク対策済み
- シングルGPU（DataParallel不使用）
- batch=1, grad_accum=4
- Gradient Checkpointing有効
- HFキャッシュを /tmp に配置（ディスク節約）
- torch.save 安定化（レガシー形式 + tmp経由アトミック保存）
- optimizer状態は保存しない（ディスク節約）

## 🚀 使い方
1. **Settings → Internet → ON**
2. Secretsに `HF_TOKEN` を設定（HFアップロード用、任意）
3. 全セルを順に実行するだけ。途中で切れても再実行でHFから自動再開

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 1: OOM対策 & インポート
# ═══════════════════════════════════════════════════════════════
import os, sys, json, math, time, gc

# ★ CUDAメモリ断片化防止（torch import より前に必須！）
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
# ★ HFキャッシュを /tmp に（/kaggle/working のディスク節約）
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.makedirs('/tmp/hf_cache', exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer

print("=" * 60)
print("🔍 リソース診断")
print("=" * 60)

# GPU
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"✅ GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")
    device = 'cuda'
    USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
else:
    print("❌ GPUなし")
    device = 'cpu'
    USE_BF16 = False

# ディスク空き確認
stat = os.statvfs('/kaggle/working')
free_gb = stat.f_frsize * stat.f_bavail / 1e9
print(f"💾 /kaggle/working 空き: {free_gb:.1f} GB")
if free_gb < 3:
    print("⚠️  空き不足！古いチェックポイントを削除します...")
    import shutil
    for d in ['checkpoints', 'downloaded_models', 'data']:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)
    os.makedirs('downloaded_models/tinyllm-nano', exist_ok=True)

import psutil
ram = psutil.virtual_memory()
print(f"📀 RAM: {ram.total/1e9:.1f} GB (free: {ram.available/1e9:.1f} GB)")
print(f"⚙️  BF16={'YES' if USE_BF16 else 'NO (FP16)'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 2: 安全保存関数 & パッケージ
# ═══════════════════════════════════════════════════════════════

def safe_save(obj, path, max_retries=3):
    """Kaggleで安全に保存（レガシーpickle + tmp経由アトミック + リトライ）"""
    tmp_path = path + '.tmp'
    for i in range(max_retries):
        try:
            torch.save(obj, tmp_path, _use_new_zipfile_serialization=False)
            os.replace(tmp_path, path)
            return True
        except RuntimeError as e:
            print(f"⚠️  Save retry {i+1}/{max_retries}: {e}")
            if os.path.exists(tmp_path): os.remove(tmp_path)
            time.sleep(2)
    raise RuntimeError("Save failed after retries")

# 不足パッケージインストール
import subprocess, importlib
for pkg in ['transformers', 'datasets', 'huggingface_hub', 'tqdm']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ 準備完了")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 3: 学習設定
# ═══════════════════════════════════════════════════════════════

HF_REPO = "Ryo3desu/tinyllm-models"
HF_MODEL_PATH = "tinyllm-nano/model.pt"
SEQ_LEN = 1024
OUTPUT_DIR = "checkpoints"

TRAIN_CONFIG = {
    'max_steps': 50000,       # ★ 総目標（大きく設定OK、中断再開対応）
    'batch_size': 1,          # T4 OOM回避
    'grad_accum': 4,          # 実効バッチ = 1×4 = 4
    'learning_rate': 3e-4,
    'warmup_steps': 100,
    'max_lr': 3e-4,
    'min_lr': 3e-5,
    'weight_decay': 0.1,
    'grad_clip': 1.0,
    'log_interval': 10,
    'save_interval': 1000,
    'gradient_checkpointing': True,
    'save_optimizer': False,  # ★ optimizer状態保存しない（ディスク節約）
}

DATA_LIMIT = 5_000_000

print("📋 設定:")
for k, v in TRAIN_CONFIG.items():
    print(f"   {k}: {v}")
print(f"   実効バッチ: {TRAIN_CONFIG['batch_size']}×{TRAIN_CONFIG['grad_accum']} = {TRAIN_CONFIG['batch_size']*TRAIN_CONFIG['grad_accum']}")
print(f"   Device: {device.upper()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 4: モデル定義
# ═══════════════════════════════════════════════════════════════

class TinyLLMLayer(nn.Module):
    """RMSNorm → Attention → RMSNorm → SwiGLU FFN"""
    def __init__(self, cfg):
        super().__init__()
        D = cfg['hidden_size']
        inter = cfg.get('intermediate_size', D * 11 // 4)
        self.norm1 = nn.RMSNorm(D, eps=1e-5)
        self.norm2 = nn.RMSNorm(D, eps=1e-5)
        self.q_proj = nn.Linear(D, D, bias=False)
        self.k_proj = nn.Linear(D, D, bias=False)
        self.v_proj = nn.Linear(D, D, bias=False)
        self.o_proj = nn.Linear(D, D, bias=False)
        self.gate_proj = nn.Linear(D, inter, bias=False)
        self.up_proj   = nn.Linear(D, inter, bias=False)
        self.down_proj = nn.Linear(inter, D, bias=False)

    def forward(self, x):
        r = x
        x = self.norm1(x)
        B, S, D = x.shape
        nh, hd = 16, D // 16
        q = self.q_proj(x).view(B, S, nh, hd).transpose(1, 2)
        k = self.k_proj(x).view(B, S, nh, hd).transpose(1, 2)
        v = self.v_proj(x).view(B, S, nh, hd).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = self.o_proj(a.transpose(1, 2).contiguous().view(B, S, D)) + r
        r = x
        x = self.norm2(x)
        x = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)) + r
        return x

class TinyLLMModel(nn.Module):
    """TinyLLM — 318M params"""
    def __init__(self, cfg):
        super().__init__()
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['hidden_size'])
        self.layers = nn.ModuleList([TinyLLMLayer(cfg) for _ in range(cfg['num_hidden_layers'])])
        self.norm = nn.RMSNorm(cfg['hidden_size'], eps=1e-5)
        self.lm_head = nn.Linear(cfg['hidden_size'], cfg['vocab_size'], bias=False)

    def forward(self, input_ids, labels=None):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
            return {'loss': loss, 'logits': logits}
        return {'loss': None, 'logits': logits}

print("✅ モデル定義完了")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 5: トークナイザー & データ準備
# ═══════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download

MODEL_DIR = 'downloaded_models/tinyllm-nano'
os.makedirs(MODEL_DIR, exist_ok=True)

# HFからconfigとtokenizerを取得（軽量）
for fname in ['tokenizer.json', 'tokenizer_config.json', 'config.json']:
    local = f'{MODEL_DIR}/{fname}'
    if not os.path.exists(local):
        print(f"📥 {fname}...")
        hf_hub_download(HF_REPO, f'tinyllm-nano/{fname}',
                       local_dir='downloaded_models', cache_dir='/tmp/hf_cache')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or '</s>'
VOCAB_SIZE = len(tokenizer)
print(f"✅ Tokenizer: vocab={VOCAB_SIZE}")

# ── データセット ──
class TokenBinDataset(IterableDataset):
    def __init__(self, path, seq_len, vocab_size):
        self.data = np.memmap(path, dtype=np.int32, mode='r')
        self.seq_len = seq_len
        self.vocab_size = vocab_size
    def __iter__(self):
        while True:
            off = np.random.randint(0, len(self.data) - self.seq_len - 1)
            tok = self.data[off:off + self.seq_len + 1]
            inp = torch.from_numpy(tok[:self.seq_len].astype(np.int64))
            lbl = torch.from_numpy(tok[1:self.seq_len + 1].astype(np.int64))
            mask = (inp >= 0) & (inp < self.vocab_size)
            lbl[~mask] = -100
            yield {'input_ids': inp, 'labels': lbl}

os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.bin'):
    print("📦 データセット準備中...")
    from datasets import load_dataset
    try:
        ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True).take(10000)
    except Exception:
        print("   CodeParrot失敗 → wikitextで代用")
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True).take(10000)
    
    all_tokens = []
    for s in ds:
        text = s.get('content') or s.get('text') or ''
        if len(text) < 10: continue
        all_tokens.extend(tokenizer.encode(text))
        if len(all_tokens) >= DATA_LIMIT: break
    
    tokens = np.array(all_tokens, dtype=np.int32)
    split = int(len(tokens) * 0.9)
    tokens[:split].tofile('data/train.bin')
    tokens[split:].tofile('data/val.bin')
    print(f"✅ train={split:,}, val={len(tokens)-split:,} tokens")
else:
    data = np.memmap('data/train.bin', dtype=np.int32, mode='r')
    print(f"✅ 既存データ: {len(data):,} tokens")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 6: モデルロード（HFから自動再開）
# ═══════════════════════════════════════════════════════════════

cfg_dict = {
    'hidden_size': 1024,
    'num_hidden_layers': 24,
    'vocab_size': VOCAB_SIZE,
    'intermediate_size': 2816,
}

model = TinyLLMModel(cfg_dict)
start_step = 0

# ★ HFから最新モデルをダウンロード → 自動再開
try:
    print("📥 HFからモデル取得中...")
    ckpt_path = hf_hub_download(HF_REPO, HF_MODEL_PATH,
                                local_dir='downloaded_models',
                                cache_dir='/tmp/hf_cache')
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
    else:
        sd = ckpt
    # module. プレフィックス除去
    sd = {k.removeprefix('module.'): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    start_step = ckpt.get('step', 0)
    print(f"✅ step {start_step:,} から再開")
except Exception as e:
    print(f"⚠️  HF未検出 ({type(e).__name__})。スクラッチ開始。")

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 {total_params/1e6:.0f}M params, from_step={start_step:,}")
model = model.to(device)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 7: Gradient Checkpointing（VRAM ~30%削減）
# ═══════════════════════════════════════════════════════════════

if TRAIN_CONFIG.get('gradient_checkpointing', False):
    from torch.utils.checkpoint import checkpoint
    for layer in model.layers:
        orig = layer.forward
        layer._original_forward = orig
        layer.forward = lambda x, l=layer: checkpoint(l._original_forward, x, use_reentrant=False)
    print("🧠 Gradient checkpointing: ON")
else:
    print("🧠 Gradient checkpointing: OFF")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 8: トレーニング
# ═══════════════════════════════════════════════════════════════

# Optimizer
try:
    optimizer = AdamW(model.parameters(), lr=TRAIN_CONFIG['learning_rate'],
                     weight_decay=TRAIN_CONFIG['weight_decay'],
                     betas=(0.9, 0.95), eps=1e-8, fused=True)
except (TypeError, RuntimeError):
    optimizer = AdamW(model.parameters(), lr=TRAIN_CONFIG['learning_rate'],
                     weight_decay=TRAIN_CONFIG['weight_decay'],
                     betas=(0.9, 0.95), eps=1e-8)

# Scheduler
def lr_lambda(step):
    wu, mx = TRAIN_CONFIG['warmup_steps'], TRAIN_CONFIG['max_steps']
    if step < wu:
        return float(step) / max(1, wu)
    p = float(step - wu) / max(1, mx - wu)
    r = TRAIN_CONFIG['min_lr'] / TRAIN_CONFIG['max_lr']
    return r + (1 - r) * 0.5 * (1 + math.cos(math.pi * p))

scheduler = LambdaLR(optimizer, lr_lambda)
for _ in range(start_step):
    scheduler.step()

scaler = torch.cuda.amp.GradScaler()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# DataLoader
train_dataset = TokenBinDataset('data/train.bin', SEQ_LEN, VOCAB_SIZE)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG['batch_size'])

# ★ シングルGPU（DataParallelはOOM原因なので使わない）
model.train()
data_iter = iter(train_loader)
global_step = start_step
total_loss = 0.0
start_time = time.time()
gc_val = TRAIN_CONFIG['grad_clip']
ga = TRAIN_CONFIG['grad_accum']

remaining = TRAIN_CONFIG['max_steps'] - start_step
print("=" * 60)
print(f"🚀 step {start_step:,} → {TRAIN_CONFIG['max_steps']:,} ({remaining:,} steps)")
print(f"   Device: {device.upper()}, AMP: {'BF16' if USE_BF16 else 'FP16'}")
print(f"   Batch={TRAIN_CONFIG['batch_size']}, GradAccum={ga}")
print("=" * 60)

while global_step < TRAIN_CONFIG['max_steps']:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs['loss']
    loss = loss / ga

    if USE_BF16:
        loss.backward()
    else:
        scaler.scale(loss).backward()
    total_loss += loss.item()

    if (global_step + 1) % ga == 0:
        if USE_BF16:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gc_val)
            optimizer.step()
        else:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), gc_val)
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        optimizer.zero_grad()

    global_step += 1

    if global_step % TRAIN_CONFIG['log_interval'] == 0:
        avg_loss = total_loss / TRAIN_CONFIG['log_interval']
        elapsed = time.time() - start_time
        tps = (global_step - start_step) * TRAIN_CONFIG['batch_size'] * SEQ_LEN / elapsed if elapsed > 0 else 0
        mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
        print(f"   {global_step:>6,} | loss={avg_loss:.4f} | {tps:>5.0f} tok/s | GPU={mem:.1f}GB | {elapsed:.0f}s")
        total_loss = 0.0

    if global_step % TRAIN_CONFIG['save_interval'] == 0:
        ckpt_dir = f'{OUTPUT_DIR}/step_{global_step}'
        os.makedirs(ckpt_dir, exist_ok=True)
        ckpt = {
            'model_state_dict': model.state_dict(),
            'config': cfg_dict,
            'step': global_step,
        }
        if TRAIN_CONFIG['save_optimizer']:
            ckpt['optimizer_state_dict'] = optimizer.state_dict()
        safe_save(ckpt, f'{ckpt_dir}/model.pt')
        print(f"💾 {ckpt_dir}/")

elapsed = time.time() - start_time
print(f"\n✅ {elapsed:.0f}s, step {start_step:,}→{global_step:,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 9: 最終保存
# ═══════════════════════════════════════════════════════════════

final_dir = f'{OUTPUT_DIR}/final'
os.makedirs(final_dir, exist_ok=True)

ckpt = {
    'model_state_dict': model.state_dict(),
    'config': cfg_dict,
    'step': global_step,
}
if TRAIN_CONFIG['save_optimizer']:
    ckpt['optimizer_state_dict'] = optimizer.state_dict()

safe_save(ckpt, f'{final_dir}/model.pt')
tokenizer.save_pretrained(final_dir)
with open(f'{final_dir}/config.json', 'w') as f:
    json.dump(cfg_dict, f, indent=2)

size_mb = os.path.getsize(f'{final_dir}/model.pt') / 1e6
print(f"💾 {final_dir}/ ({size_mb:.0f} MB, step {global_step:,})")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 10: HF にアップロード
# ═══════════════════════════════════════════════════════════════

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print("✅ SecretsからHF_TOKEN取得")
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print("⚠️  HF_TOKEN未設定。スキップ。")
    print("   設定: Kaggle → Add-ons → Secrets → HF_TOKEN")
else:
    try:
        from huggingface_hub import login, HfApi
        login(token=hf_token)
        api = HfApi()
        gb = os.path.getsize(f'{final_dir}/model.pt')/1e9
        print(f"📤 Uploading {gb:.2f} GB...")
        api.upload_file(
            path_or_fileobj=f'{final_dir}/model.pt',
            path_in_repo=HF_MODEL_PATH,
            repo_id=HF_REPO, repo_type="model",
        )
        print("✅ 次回実行で自動再開されます")
    except Exception as e:
        print(f"❌ {e}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 11: 簡易推論テスト
# ═══════════════════════════════════════════════════════════════

@torch.no_grad()
def generate(prompt, max_tokens=128, temperature=0.7):
    model.eval()
    ids = tokenizer.encode(prompt)
    if not ids: ids = [tokenizer.bos_token_id or 0]
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_tokens):
        if inp.size(1) > SEQ_LEN: inp = inp[:, -SEQ_LEN:]
        logits = model(inp)['logits'][0, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        if nxt == tokenizer.eos_token_id: break
        inp = torch.cat([inp, torch.tensor([[nxt]], device=device)], dim=1)
    return tokenizer.decode(inp[0].tolist(), skip_special_tokens=True)

prompt = "def fibonacci(n):"
print(f"🧪 {prompt}")
print(f"🤖 {generate(prompt)[:200]}")

## ✅ 完了

- 学習済みモデル: `checkpoints/final/`
- HF_TOKEN があれば自動アップロード済み
- **さらに学習を続けるには、このノートブックをもう一度最初から実行するだけ**
- HF から最新のチェックポイントが自動で読み込まれ、続きから再開します